# 포트폴리오 결정이론 — Colab 실습 노트북

`portfolio_models_edu.py`를 **Google Colab**에서 셀 단위로 실행할 수 있도록 재구성한 노트북입니다.

기대수익률을 쓰지 않는 **위험(Risk) 기반 자산배분 4대 모델**을 직접 계산하고 시각화합니다.

| 모델 | 목적 |
|---|---|
| **Equal-Weight** | $w_i = 1/n$ (벤치마크) |
| **GMV** | 포트폴리오 분산 최소화 $\min\ w^T\Sigma w$ |
| **MDP** | 분산비율 최대화 $\max\ (w^T\sigma)/\sigma_p$ |
| **Risk Parity** | 위험기여도 균등 $RC_i = 1/n$ |

> 사용법: 위에서부터 셀을 순서대로 실행(Shift+Enter)하세요.

## 0. 환경 준비
Colab에는 numpy/pandas/matplotlib가 기본 설치돼 있습니다. 엑셀 로드를 위해 openpyxl만 확인합니다.

In [ ]:
# 필요 패키지 (Colab에는 대부분 이미 설치됨)
!pip install -q openpyxl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True
print('준비 완료:', np.__version__, pd.__version__)

## 1. 핵심 함수 정의
제약조건 투영 · 투영 경사하강법 · 4대 모델 · 성과지표를 한 셀에 정의합니다.
각 함수는 수식과 1:1로 대응합니다.

In [ ]:
# ---------- 공통: 제약조건 투영 + 최적화 엔진 ----------
def project_weights(weights, lower, upper):
    """비중을 [lower, upper]로 clip 후 합계=1로 정규화 (투영)."""
    weights = np.clip(weights, lower, upper)
    return weights / np.sum(weights)

def auto_constraints(n):
    """종목 수 n에 따른 동적 제약조건: 균등배분의 50%~300%."""
    return max(0.01, 0.5 / n), min(0.40, 3.0 / n)

def projected_gradient_descent(objective, gradient, n, lower, upper,
                               n_restarts=30, learning_rate=0.01,
                               max_iter=1000, tol=1e-6, seed=0):
    """투영 경사하강법 + multi-start. objective를 '최소화'한다."""
    rng = np.random.default_rng(seed)
    best_w, best_obj = None, float('inf')
    for _ in range(n_restarts):
        w = rng.uniform(lower, upper, n); w = w / np.sum(w)
        prev_obj, lr = float('inf'), learning_rate
        for _ in range(max_iter):
            w = w - lr * gradient(w)           # 1. 하강
            w = project_weights(w, lower, upper)  # 2. 투영
            curr = objective(w)
            if abs(curr - prev_obj) < tol:     # 3. 수렴
                break
            prev_obj = curr; lr *= 0.999       # 4. 학습률 감소
        if curr < best_obj:
            best_obj, best_w = curr, w
    return best_w, best_obj

In [ ]:
# ---------- 모델 ① Equal-Weight ----------
def equal_weight(cov):
    n = len(cov)
    return np.ones(n) / n

# ---------- 모델 ② GMV (최소분산) ----------
def gmv_weights(cov, lower=None, upper=None, seed=0):
    n = len(cov); S = np.asarray(cov, float)
    if lower is None or upper is None: lower, upper = auto_constraints(n)
    obj = lambda w: float(w.T @ S @ w)          # wᵀΣw
    grad = lambda w: 2 * (S @ w)                # 2Σw
    w, _ = projected_gradient_descent(obj, grad, n, lower, upper, seed=seed)
    return w

# ---------- 모델 ③ MDP (최대분산투자) ----------
def mdp_weights(cov, lower=None, upper=None, seed=0):
    n = len(cov); S = np.asarray(cov, float); vols = np.sqrt(np.diag(S))
    if lower is None or upper is None: lower, upper = auto_constraints(n)
    def obj(w):
        pv = np.sqrt(w.T @ S @ w)
        return float('inf') if pv < 1e-10 else -float(w @ vols) / pv
    def grad(w):
        pv = np.sqrt(w.T @ S @ w)
        if pv < 1e-10: return np.zeros(n)
        wv = w @ vols; gpv = (S @ w) / pv
        return -(vols * pv - wv * gpv) / (pv ** 2)
    w, _ = projected_gradient_descent(obj, grad, n, lower, upper, seed=seed)
    return w

def diversification_ratio(w, cov):
    S = np.asarray(cov, float); vols = np.sqrt(np.diag(S))
    return float(w @ vols) / float(np.sqrt(w.T @ S @ w))

# ---------- 모델 ④ Risk Parity (위험균등) ----------
def risk_contributions(w, cov):
    S = np.asarray(cov, float); pv = np.sqrt(w.T @ S @ w)
    if pv < 1e-10: return np.zeros(len(w))
    return np.multiply(S @ w, w) / pv          # w_i (Σw)_i / σ_p

def risk_parity_weights(cov, lower=None, upper=None, seed=0):
    n = len(cov); S = np.asarray(cov, float)
    if lower is None or upper is None: lower, upper = auto_constraints(n)
    target = 1.0 / n
    def obj(w):
        rc = risk_contributions(w, S); pct = rc / rc.sum()  # 퍼센트 기여도(합=1)
        return float(np.sum((pct - target) ** 2))
    def grad(w):
        eps = 1e-8; g = np.zeros(n)
        for i in range(n):
            h = np.zeros(n); h[i] = eps
            g[i] = (obj(w + h) - obj(w - h)) / (2 * eps)
        return g
    w, _ = projected_gradient_descent(obj, grad, n, lower, upper, seed=seed)
    return w

In [ ]:
# ---------- 성과지표 + 유틸 ----------
def performance_metrics(weights, returns_df, periods_per_year=252):
    pr = returns_df.dot(weights)
    cum = (1 + pr).cumprod()
    total = cum.iloc[-1] - 1; nd = len(returns_df)
    ann_ret = (1 + total) ** (periods_per_year / nd) - 1
    ann_vol = pr.std() * np.sqrt(periods_per_year)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0.0
    mdd = ((cum - cum.cummax()) / cum.cummax()).min()
    return {'연환산수익률': ann_ret, '연환산변동성': ann_vol,
            '샤프비율': sharpe, '최대낙폭(MDD)': mdd}

def make_cov(vols, corr):
    """변동성 벡터 + 상관행렬 → 공분산 행렬 (Σ_ij = σ_i σ_j ρ_ij)."""
    vols = np.asarray(vols, float); D = np.diag(vols)
    return D @ np.asarray(corr, float) @ D

def all_models(cov, lower=None, upper=None):
    return {
        'Equal-Weight': equal_weight(cov),
        'GMV': gmv_weights(cov, lower, upper),
        'MDP': mdp_weights(cov, lower, upper),
        'Risk Parity': risk_parity_weights(cov, lower, upper),
    }

print('함수 정의 완료 ✔')

## 2. 데모 — 3종목 예제
변동성과 상관관계가 서로 다른 자산 A/B/C로 4대 모델의 비중과 위험기여도를 비교합니다.

In [ ]:
tickers = ['A', 'B', 'C']
vols = [0.10, 0.20, 0.15]        # A=저변동, B=고변동
corr = [[1.0, 0.2, 0.5],
        [0.2, 1.0, 0.3],
        [0.5, 0.3, 1.0]]
cov = make_cov(vols, corr)

# 데모는 제약을 느슨하게(5%~60%) 주어 각 모델 성향이 뚜렷하게 드러나도록 함
models = all_models(cov, lower=0.05, upper=0.60)

df = pd.DataFrame(models, index=tickers)
print('=== 비중 (weights) ===')
display(df.style.format('{:.1%}'))

print('\n=== 퍼센트 위험기여도 (Risk Contribution %) ===')
rc = {name: risk_contributions(w, cov) / risk_contributions(w, cov).sum()
      for name, w in models.items()}
rc_df = pd.DataFrame(rc, index=tickers)
display(rc_df.style.format('{:.1%}'))

print('분산비율(DR):', {k: round(diversification_ratio(v, cov), 3) for k, v in models.items()})

### 2-1. 시각화 — 비중 & 위험기여도

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4))
df.T.plot(kind='bar', ax=axL)
axL.set_title('Portfolio Weights'); axL.set_ylabel('Weight')
axL.legend(tickers, title='Asset'); axL.tick_params(axis='x', rotation=0)

rc_df.T.plot(kind='bar', ax=axR)
axR.axhline(1/len(tickers), ls='--', color='gray', label='target 1/n')
axR.set_title('Risk Contribution (%)'); axR.set_ylabel('RC share')
axR.legend(); axR.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

print('관찰: GMV는 저변동성 A에 쏠리고, Risk Parity는 RC%가 1/3에 수렴하며',
      '고변동성 B의 비중이 작아진다.')

## 3. 상관관계 실험 — MDP와 분산비율
두 자산 A·B의 상관계수 $\rho$를 바꿔가며 **분산비율(DR)** 이 어떻게 변하는지 관찰합니다.
(변동성을 동일하게 두어 순수하게 상관효과만 봅니다.)

In [ ]:
rhos = np.linspace(-0.9, 0.9, 19)
drs = []
for rho in rhos:
    c = make_cov([0.15, 0.15], [[1.0, rho], [rho, 1.0]])
    w = mdp_weights(c)
    drs.append(diversification_ratio(w, c))

plt.figure(figsize=(8, 4))
plt.plot(rhos, drs, 'o-')
plt.xlabel('상관계수 ρ(A,B)'); plt.ylabel('분산비율 DR')
plt.title('상관관계가 낮을수록 분산효과(DR)가 커진다')
plt.axhline(1.0, ls='--', color='gray')
plt.show()

for rho in [0.9, 0.0, -0.5]:
    c = make_cov([0.15, 0.15], [[1.0, rho], [rho, 1.0]])
    w = mdp_weights(c)
    print(f'ρ={rho:+.1f} → MDP 비중 A={w[0]:.1%}, B={w[1]:.1%}, DR={diversification_ratio(w, c):.3f}')

## 4. 실제 데이터로 실행 (엑셀 업로드)
본인의 가격 데이터 엑셀 파일을 업로드해 4대 모델을 최적화하고 성과를 비교합니다.

**지원 형식**
- 세로형(Long): `Date`, `Ticker`, `Price` 컬럼
- 가로형(Wide): 첫 컬럼이 날짜 + 종목별 가격 컬럼들

In [ ]:
def load_excel_returns(path):
    """엑셀 → (일간수익률 DataFrame, tickers). 세로형/가로형 자동 감지."""
    df = pd.read_excel(path)
    df.columns = [str(c).strip() for c in df.columns]
    low = [c.lower() for c in df.columns]
    if {'date', 'ticker', 'price'}.issubset(set(low)):
        rn = {c: c.lower() for c in df.columns if c.lower() in ('date','ticker','price')}
        df = df.rename(columns=rn)
        df['date'] = pd.to_datetime(df['date'])
        df['price'] = pd.to_numeric(df['price'], errors='coerce')
        df = df.dropna(subset=['date','ticker','price'])
        pivot = df.pivot_table(index='date', columns='ticker', values='price', aggfunc='last')
    else:
        dc = df.columns[0]
        df[dc] = pd.to_datetime(df[dc]); df = df.set_index(dc)
        pivot = df.select_dtypes(include=[np.number])
    pivot = pivot.ffill().bfill().dropna()
    ret = pivot.pct_change().dropna().replace([np.inf, -np.inf], np.nan).dropna()
    return ret, list(ret.columns)

In [ ]:
# Colab 파일 업로드
from google.colab import files
uploaded = files.upload()   # 엑셀 파일 선택
path = next(iter(uploaded))
print('업로드됨:', path)

returns, tickers = load_excel_returns(path)
cov = returns.cov().values
print(f'{len(returns)}일, {len(tickers)}종목:', tickers)

In [ ]:
# 4대 모델 최적화 + 성과지표
models = all_models(cov)   # 실제 데이터는 자동 제약조건 사용

w_df = pd.DataFrame(models, index=tickers)
print('=== 비중 ===')
display(w_df.style.format('{:.2%}'))

rows = {name: performance_metrics(w, returns) for name, w in models.items()}
perf = pd.DataFrame(rows).T
print('\n=== 성과지표 ===')
display(perf.style.format({'연환산수익률':'{:.2%}','연환산변동성':'{:.2%}',
                           '샤프비율':'{:.2f}','최대낙폭(MDD)':'{:.2%}'}))

In [ ]:
# 누적수익률 & 위험-수익 프로파일
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
for name, w in models.items():
    cum = (1 + returns.dot(w)).cumprod()
    a1.plot(cum.index, cum.values, label=name)
a1.set_title('Cumulative Returns'); a1.legend(); a1.tick_params(axis='x', rotation=45)

for name, w in models.items():
    m = performance_metrics(w, returns)
    a2.scatter(m['연환산변동성'], m['연환산수익률'], s=150, edgecolors='black')
    a2.annotate(name, (m['연환산변동성'], m['연환산수익률']),
                textcoords='offset points', xytext=(8, 5))
a2.set_xlabel('Annual Volatility'); a2.set_ylabel('Annual Return')
a2.set_title('Risk-Return Profile')
plt.tight_layout(); plt.show()

---
### 참고 / 주의점 (강의 토론용)
- **샤프비율**은 무위험이자율을 0으로 가정합니다.
- **인샘플 최적화**: 같은 기간으로 최적화·평가하므로 과적합 위험 → 워크포워드 백테스트 권장.
- **공분산 추정**이 불안정하면(종목 많음) Ledoit-Wolf 축소추정 고려.
- 이 노트북은 라이브러리 없이 최적화 원리를 보여주기 위한 **교육용**입니다.